This notebook is for running the LLM as a judge scoring for the OSQs. 

Simple Correct/Incorrect and Scoring with a Rubric are to be used.

Framework:
- Pull in the OSQ response data
- (Optional) list of judge models -- default only CXhatgpt5 or Chatgpt4o-mini
- List of Prompts
  - Binary Correct/Incorrect
  - Scoring
    - Rubric 1 (from the MCQ -> OSQ conversion)
    - Rubric 2 (outside of recommended from conversion)

# Configuration

In [18]:
import os
import json
from pathlib import Path
from openai import OpenAI
from datetime import datetime
from statistics import mean
from rich.console import Console
from rich.table import Table

console = Console()

# -----------------------------
# CONFIGURATION
# -----------------------------
MODEL = "openai/gpt-5"                    # OpenRouter judge model
SAMPLE_N = 5                              # 0 = all samples, else N for quick test
MAX_TOKENS = 2000
TEMPERATURE = 0.0


# Directory structure
BASE_DIR = Path("../phase4_inference/downloaded_output")  # go up once to reach phase4
TASK_NAME = "sysengbench-osq"
TASK_DIR = BASE_DIR / TASK_NAME
OUTPUT_DIR = Path("judge_results")  # stays local to phase5_llm_as_a_judge
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)
console.print(f"[green]OPENROUTER_API_KEY present:[/green] {bool(os.getenv('OPENROUTER_API_KEY'))}")

OPENROUTER_API_KEY present: True

In [19]:
print("BASE_DIR:", BASE_DIR.resolve())
print("TASK_DIR:", TASK_DIR.resolve())


BASE_DIR: C:\Users\rabel\Desktop\dissertation\src\phase4_inference\downloaded_output
TASK_DIR: C:\Users\rabel\Desktop\dissertation\src\phase4_inference\downloaded_output\sysengbench-osq


# Prompt Template

In [22]:
# -----------------------------
# PROMPT TEMPLATE (100-point scale)
# -----------------------------
JUDGE_PROMPT = """You are an expert systems engineering educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of systems engineering concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
"OUTPUT FORMAT (strict JSON):
{{
  \"technical_accuracy\": {{\"score\": <0–20>, \"justification\": \"<1–2 sentences>\"}},
  \"conceptual_understanding\": {{\"score\": <0–20>, \"justification\": \"<1–2 sentences>\"}},
  \"completeness\": {{\"score\": <0–20>, \"justification\": \"<1–2 sentences>\"}},
  \"clarity_organization\": {{\"score\": <0–20>, \"justification\": \"<1–2 sentences>\"}},
  \"professional_relevance\": {{\"score\": <0–20>, \"justification\": \"<1–2 sentences>\"}},
  \"overall_score\": <0–100>,
  \"overall_assessment\": \"<summary>\",
  \"key_strengths\": \"<strengths>\",
  \"improvement_areas\": \"<areas for improvement>\"
}}

"""

In [23]:
# -----------------------------
# UTILITIES
# -----------------------------
def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def find_sample_and_results(model_dir: Path):
    """Find the newest sample + results pair inside a model directory."""
    samples = sorted(model_dir.glob("samples_*.jsonl"), key=os.path.getmtime, reverse=True)
    results = sorted(model_dir.glob("results_*.json"), key=os.path.getmtime, reverse=True)
    if not samples or not results:
        return None, None
    return samples[0], results[0]

def judge_sample(sample, idx):
    prompt = JUDGE_PROMPT.format(
        osq_question=sample.get("osq_question", ""),
        expected_answer=sample.get("expected_answer", ""),
        student_response=sample.get("student_response", ""),
        blooms_level=sample.get("blooms_level", "N/A"),
        se_domain=sample.get("se_domain", "General Systems Engineering"),
    )
    try:
        completion = client.chat.completions.create(
            model=MODEL,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            messages=[
                {"role": "system", "content": "You are an expert evaluator in systems engineering education."},
                {"role": "user", "content": prompt}
            ]
        )
        raw_output = completion.choices[0].message.content.strip()
        parsed = json.loads(raw_output)
    except Exception as e:
        parsed = {"error": str(e)}
    parsed["sample_id"] = idx
    parsed["timestamp"] = datetime.now().isoformat()
    return parsed

def aggregate_scores(evaluations):
    scores = {k: [] for k in [
        "technical_accuracy", "conceptual_understanding",
        "completeness", "clarity_organization",
        "professional_relevance", "overall_score"
    ]}
    for e in evaluations:
        if "error" not in e:
            for k in scores:
                val = e.get(k, {}).get("score") if isinstance(e.get(k), dict) else e.get(k)
                if isinstance(val, (int, float)):
                    scores[k].append(val)
    return {k: round(mean(v), 3) for k, v in scores.items() if v}

# -----------------------------
# DISCOVER ALL MODELS
# -----------------------------
if not TASK_DIR.exists():
    raise FileNotFoundError(f"Task directory not found: {TASK_DIR}")

model_dirs = [d for d in TASK_DIR.iterdir() if d.is_dir()]
if not model_dirs:
    raise FileNotFoundError(f"No model directories found under {TASK_DIR}")

# Build Rich table of model statuses
table = Table(title=f"LLM-as-a-Judge Discovery for Task: {TASK_NAME}")
table.add_column("Model Directory")
table.add_column("Samples Found")
table.add_column("Results Found")
table.add_column("Judge Results Exist")
table.add_column("Judged Output File", overflow="fold")

model_status = []
for model_dir in model_dirs:
    sample, results = find_sample_and_results(model_dir)
    judge_files = list(OUTPUT_DIR.glob(f"*{model_dir.name}*judge.json"))
    judged = "✅" if judge_files else "❌"
    table.add_row(
        model_dir.name,
        "✅" if sample else "❌",
        "✅" if results else "❌",
        judged,
        judge_files[0].name if judge_files else "-"
    )
    model_status.append((model_dir, sample, results, judge_files))

console.print(table)

# -----------------------------
# MAIN EXECUTION LOOP
# -----------------------------
for model_dir, sample_path, results_path, judge_files in model_status:
    if not sample_path or not results_path:
        console.print(f"[yellow]Skipping {model_dir.name} — missing files[/yellow]")
        continue

    if judge_files and SAMPLE_N == 0:
        console.print(f"[cyan]Skipping {model_dir.name} (already judged)[/cyan]")
        continue

    samples = load_jsonl(sample_path)
    if SAMPLE_N > 0:
        samples = samples[:SAMPLE_N]
        console.print(f"[green]Testing only {SAMPLE_N} samples for {model_dir.name}[/green]")
    else:
        console.print(f"[green]Judging all {len(samples)} samples for {model_dir.name}[/green]")

    results_meta = load_json(results_path)
    evaluations = [judge_sample(sample, i) for i, sample in enumerate(samples)]
    aggregate = aggregate_scores(evaluations)

    # Prepare results JSON
    judged_results = results_meta.copy()
    judged_results["judge_metadata"] = {
        "judge_model": MODEL,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "scale": "0–100 (five 20-point dimensions)",
        "timestamp": datetime.now().isoformat(),
        "samples_judged": len(samples),
        "source_model": model_dir.name
    }
    judged_results["results"] = {
        TASK_NAME: {
            "mean_scores": aggregate,
            "n_evaluated": len(samples),
            "evaluation_mode": "LLM-as-a-Judge"
        }
    }

    timestamp = datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
    output_file = OUTPUT_DIR / f"results_{timestamp}_{model_dir.name}_{MODEL.replace('/', '_')}-judge.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(judged_results, f, indent=2)
    console.print(f"[bold green]✅ Saved judged results to {output_file}[/bold green]")


                      LLM-as-a-Judge Discovery for Task: sysengbench-osq                      
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Model Directory ┃ Samples Found ┃ Results Found ┃ Judge Results Exist ┃ Judged Output File ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ gemma3__27b     │ ✅            │ ✅            │ ❌                  │ -                  │
│ gemma3__4b      │ ✅            │ ✅            │ ❌                  │ -                  │
└─────────────────┴───────────────┴───────────────┴─────────────────────┴────────────────────┘

Testing only 5 samples for gemma3__27b

✅ Saved judged results to judge_results\results_2025-10-16T23-26-33_gemma3__27b_openai_gpt-5-judge.json

Testing only 5 samples for gemma3__4b

✅ Saved judged results to judge_results\results_2025-10-16T23-27-21_gemma3__4b_openai_gpt-5-judge.json

# Refactoring the code for efficiency
- Save to a file after each inference 
- Pickup where the last inference stopped in the log if something happens
- tqdm progress bar 

In [ ]:
# -----------------------------
# INCREMENTAL + RESUMABLE LLM-as-a-Judge LOOP (with tqdm)
# -----------------------------
from tqdm.auto import tqdm
import os, json
from pathlib import Path
from datetime import datetime
from statistics import mean
from rich.console import Console
from rich.table import Table
from openai import OpenAI

console = Console()

# -----------------------------
# CONFIGURATION
# -----------------------------
MODEL = "openai/gpt-5"                    # OpenRouter judge model
SAMPLE_N = 5                              # 0 = all samples, else N for quick test
MAX_TOKENS = 2000
TEMPERATURE = 0.0
TASK_NAME = "sysengbench-osq"

# Directory structure
BASE_DIR = Path("../phase4_inference/downloaded_output")  # parent folder of all model outputs
TASK_DIR = BASE_DIR / TASK_NAME
OUTPUT_DIR = Path("judge_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)
console.print(f"[green]OPENROUTER_API_KEY present:[/green] {bool(os.getenv('OPENROUTER_API_KEY'))}")

# -----------------------------
# PROMPT TEMPLATE (100-point scale)
# -----------------------------
JUDGE_PROMPT = """You are an expert systems engineering educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of systems engineering concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{{
  "technical_accuracy": {{"score": "<0–20>", "justification": "<1–2 sentences>"}},
  "conceptual_understanding": {{"score": "<0–20>", "justification": "<1–2 sentences>"}},
  "completeness": {{"score": "<0–20>", "justification": "<1–2 sentences>"}},
  "clarity_organization": {{"score": "<0–20>", "justification": "<1–2 sentences>"}},
  "professional_relevance": {{"score": "<0–20>", "justification": "<1–2 sentences>"}},
  "overall_score": "<0–100>",
  "overall_assessment": "<summary>",
  "key_strengths": "<strengths>",
  "improvement_areas": "<areas for improvement>"
}}
"""

# -----------------------------
# UTILITIES
# -----------------------------
def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def find_sample_and_results(model_dir: Path):
    samples = sorted(model_dir.glob("samples_*.jsonl"), key=os.path.getmtime, reverse=True)
    results = sorted(model_dir.glob("results_*.json"), key=os.path.getmtime, reverse=True)
    if not samples or not results:
        return None, None
    return samples[0], results[0]

def judge_sample(sample, idx):
    prompt = JUDGE_PROMPT.format(
        osq_question=sample.get("osq_question", ""),
        expected_answer=sample.get("expected_answer", ""),
        student_response=sample.get("student_response", ""),
        blooms_level=sample.get("blooms_level", "N/A"),
        se_domain=sample.get("se_domain", "General Systems Engineering"),
    )
    try:
        completion = client.chat.completions.create(
            model=MODEL,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            messages=[
                {"role": "system", "content": "You are an expert evaluator in systems engineering education."},
                {"role": "user", "content": prompt}
            ]
        )
        raw_output = completion.choices[0].message.content.strip()
        parsed = json.loads(raw_output)
    except Exception as e:
        parsed = {"error": str(e)}
    parsed["sample_id"] = idx
    parsed["timestamp"] = datetime.now().isoformat()
    return parsed

def aggregate_scores(evaluations):
    scores = {k: [] for k in [
        "technical_accuracy", "conceptual_understanding",
        "completeness", "clarity_organization",
        "professional_relevance", "overall_score"
    ]}
    for e in evaluations:
        if "error" not in e:
            for k in scores:
                val = e.get(k, {}).get("score") if isinstance(e.get(k), dict) else e.get(k)
                if isinstance(val, (int, float)):
                    scores[k].append(val)
    return {k: round(mean(v), 3) for k, v in scores.items() if v}

# -----------------------------
# DISCOVER ALL MODELS
# -----------------------------
if not TASK_DIR.exists():
    raise FileNotFoundError(f"Task directory not found: {TASK_DIR}")

model_dirs = [d for d in TASK_DIR.iterdir() if d.is_dir()]
if not model_dirs:
    raise FileNotFoundError(f"No model directories found under {TASK_DIR}")

table = Table(title=f"LLM-as-a-Judge Discovery for Task: {TASK_NAME}")
table.add_column("Model Directory")
table.add_column("Samples Found")
table.add_column("Results Found")
table.add_column("Judge Results Exist")
table.add_column("Judged Output File", overflow="fold")

model_status = []
for model_dir in model_dirs:
    sample, results = find_sample_and_results(model_dir)
    judge_files = list(OUTPUT_DIR.glob(f"*{model_dir.name}*judge.json"))
    judged = "✅" if judge_files else "❌"
    table.add_row(
        model_dir.name,
        "✅" if sample else "❌",
        "✅" if results else "❌",
        judged,
        judge_files[0].name if judge_files else "-"
    )
    model_status.append((model_dir, sample, results, judge_files))

console.print(table)

# -----------------------------
# MAIN EXECUTION LOOP (with tqdm + partial resume)
# -----------------------------
for model_dir, sample_path, results_path, judge_files in model_status:
    if not sample_path or not results_path:
        console.print(f"[yellow]Skipping {model_dir.name} — missing files[/yellow]")
        continue

    if judge_files and SAMPLE_N == 0:
        console.print(f"[cyan]Skipping {model_dir.name} (already judged)[/cyan]")
        continue

    samples = load_jsonl(sample_path)
    if SAMPLE_N > 0:
        samples = samples[:SAMPLE_N]
        console.print(f"[green]Testing only {SAMPLE_N} samples for {model_dir.name}[/green]")
    else:
        console.print(f"[green]Judging all {len(samples)} samples for {model_dir.name}[/green]")

    results_meta = load_json(results_path)

    # Partial log path
    partial_log = OUTPUT_DIR / f"{model_dir.name}_{MODEL.replace('/', '_')}_partial.jsonl"
    done_ids = set()
    if partial_log.exists():
        with open(partial_log, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    done_ids.add(obj["sample_id"])
                except Exception:
                    continue
        console.print(f"[blue]Resuming from partial log: {len(done_ids)} already done[/blue]")

    # Judge loop with tqdm
    new_evals = []
    with open(partial_log, "a", encoding="utf-8") as fout:
        for i, sample in enumerate(tqdm(samples, desc=f"Judging {model_dir.name}", unit="sample")):
            if i in done_ids:
                continue
            eval_obj = judge_sample(sample, i)
            fout.write(json.dumps(eval_obj) + "\n")
            fout.flush()
            new_evals.append(eval_obj)

    # Reload all partials for aggregation
    with open(partial_log, "r", encoding="utf-8") as f:
        all_evaluations = [json.loads(line) for line in f if line.strip()]
    aggregate = aggregate_scores(all_evaluations)

    # Prepare results JSON
    judged_results = results_meta.copy()
    judged_results["judge_metadata"] = {
        "judge_model": MODEL,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "scale": "0–100 (five 20-point dimensions)",
        "timestamp": datetime.now().isoformat(),
        "samples_judged": len(all_evaluations),
        "source_model": model_dir.name,
        "partial_file": str(partial_log)
    }
    judged_results["results"] = {
        TASK_NAME: {
            "mean_scores": aggregate,
            "n_evaluated": len(all_evaluations),
            "evaluation_mode": "LLM-as-a-Judge"
        }
    }

    timestamp = datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
    final_result = OUTPUT_DIR / f"results_{timestamp}_{model_dir.name}_{MODEL.replace('/', '_')}-judge.json"
    with open(final_result, "w", encoding="utf-8") as f:
        json.dump(judged_results, f, indent=2)
    console.print(f"[bold green]✅ Saved judged results to {final_result}[/bold green]")

    # Archive partial file
    archive_dir = OUTPUT_DIR / "partials_archive"
    archive_dir.mkdir(exist_ok=True)
    archived_path = archive_dir / f"{partial_log.name}.done"
    partial_log.rename(archived_path)
    console.print(f"[cyan]Archived partial log to {archived_path}[/cyan]")


# Adding in the actual judges samples... then use that to create the results.json files.

In [24]:
from pathlib import Path
import json
from datetime import datetime
from tqdm.auto import tqdm

# --- CONFIG ---
TASK_NAME = "sysengbench-osq"
MODEL = "gemma3:27b"                    # source model
JUDGE_MODEL = "openai/gpt-5"            # judge
SAMPLE_N = 2                            # limit (0 = all)
BASE_DIR = Path("../phase4_inference/downloaded_output") / TASK_NAME
OUTPUT_DIR = Path("phase5_llm_as_a_judge")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Locate newest samples file ---
model_dir = next(d for d in BASE_DIR.iterdir() if MODEL.replace(":", "__") in d.name)
samples_path = sorted(model_dir.glob("samples_*.jsonl"), key=lambda f: f.stat().st_mtime, reverse=True)[0]
samples = [json.loads(line) for line in open(samples_path, encoding="utf-8")]
if SAMPLE_N > 0:
    samples = samples[:SAMPLE_N]

# --- Build judge-ready JSONL ---
timestamp = datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
judged_jsonl = OUTPUT_DIR / f"samples_{timestamp}_{model_dir.name}_{JUDGE_MODEL.replace('/', '_')}-judge.jsonl"
partial_jsonl = OUTPUT_DIR / f"samples_{timestamp}_{model_dir.name}_{JUDGE_MODEL.replace('/', '_')}-partial.jsonl"

template = {
    "technical_accuracy": {"score": None, "justification": None},
    "conceptual_understanding": {"score": None, "justification": None},
    "completeness": {"score": None, "justification": None},
    "clarity_organization": {"score": None, "justification": None},
    "professional_relevance": {"score": None, "justification": None},
    "overall_score": None,
    "overall_assessment": None,
    "key_strengths": None,
    "improvement_areas": None,
}

with open(judged_jsonl, "w", encoding="utf-8") as f_full, open(partial_jsonl, "w", encoding="utf-8") as f_partial:
    for i, s in enumerate(tqdm(samples, desc=f"Preparing judge samples for {model_dir.name}")):
        entry = {
            "sample_id": i,
            "osq_question": s.get("osq_question", ""),
            "expected_answer": s.get("expected_answer", ""),
            "student_response": s.get("student_response", ""),
            "blooms_level": s.get("blooms_level", ""),
            "se_domain": s.get("se_domain", ""),
            "judge_fields": template.copy(),
        }
        f_full.write(json.dumps(entry) + "\n")
        f_partial.write(json.dumps(entry) + "\n")

print(f"\n✅ Created judge sample files:\n  Full: {judged_jsonl}\n  Partial: {partial_jsonl}")


Preparing judge samples for gemma3__27b:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Created judge sample files:
  Full: phase5_llm_as_a_judge\samples_2025-10-16T23-32-45_gemma3__27b_openai_gpt-5-judge.jsonl
  Partial: phase5_llm_as_a_judge\samples_2025-10-16T23-32-45_gemma3__27b_openai_gpt-5-partial.jsonl


In [ ]:
import json
from statistics import mean

judged_path = "phase5_llm_as_a_judge/samples_...-judge.jsonl"
entries = [json.loads(l) for l in open(judged_path, encoding="utf-8")]

scores = {k: [] for k in [
    "technical_accuracy", "conceptual_understanding",
    "completeness", "clarity_organization",
    "professional_relevance", "overall_score"
]}

for e in entries:
    j = e["judge_fields"]
    for k in scores:
        val = j[k]["score"] if isinstance(j[k], dict) else j[k]
        if isinstance(val, (int, float)):
            scores[k].append(val)

aggregate = {k: round(mean(v), 3) for k, v in scores.items() if v}
print(aggregate)


## attempt 2

In [25]:
# ================================================
# Phase 5: Build Judge Samples + Incremental Judging (Resumable, with tqdm)
# ================================================
from pathlib import Path
from datetime import datetime
from statistics import mean
from collections import OrderedDict
from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
from openai import OpenAI
import os, json, copy

console = Console()

# -----------------------------
# CONFIG
# -----------------------------
TASK_NAME = "sysengbench-osq"

# OpenRouter model/router + run params
JUDGE_MODEL = "openai/gpt-5"
TEMPERATURE = 0.0
MAX_TOKENS = 2000

# Control how many samples to include in Stage 1 and to judge:
#  - 0 means ALL samples
#  - e.g., 5 for a quick test
SAMPLE_N = 2

# Path layout (relative to this notebook directory: src/phase5_llm_as_a_judge/)
PHASE4_DIR  = Path("../phase4_inference/downloaded_output") / TASK_NAME
SAMPLES_OUT = Path("judge_samples")   # Stage 1 outputs (full + partial per model+run)
RESULTS_OUT = Path("judge_results")   # Stage 2 aggregates (judge results)
SAMPLES_OUT.mkdir(parents=True, exist_ok=True)
RESULTS_OUT.mkdir(parents=True, exist_ok=True)

# Initialize OpenRouter client
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)
console.print(f"[green]OPENROUTER_API_KEY present:[/green] {bool(os.getenv('OPENROUTER_API_KEY'))}")

# -----------------------------
# JUDGE PROMPT (0–100 via five 0–20 dims) – braces escaped for .format()
# -----------------------------
JUDGE_PROMPT = """You are an expert systems engineering educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of systems engineering concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{{
  "technical_accuracy": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "conceptual_understanding": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "completeness": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "clarity_organization": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "professional_relevance": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "overall_score": <0–100>,
  "overall_assessment": "<summary>",
  "key_strengths": "<strengths>",
  "improvement_areas": "<areas for improvement>"
}}
"""

# -----------------------------
# Utilities
# -----------------------------
def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def write_jsonl(path: Path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

def newest(glob_iter):
    items = sorted(list(glob_iter), key=lambda p: p.stat().st_mtime, reverse=True)
    return items[0] if items else None

def aggregate_scores(evals_dict_by_id):
    scores = {k: [] for k in [
        "technical_accuracy", "conceptual_understanding",
        "completeness", "clarity_organization",
        "professional_relevance", "overall_score"
    ]}
    for e in evals_dict_by_id.values():
        jf = e.get("judge_fields", {})
        for k in scores:
            val = jf.get(k, {}).get("score") if isinstance(jf.get(k), dict) else jf.get(k)
            if isinstance(val, (int, float)):
                scores[k].append(val)
    return {k: round(mean(v), 3) for k, v in scores.items() if v}

def safe_parse_json(s):
    try:
        return json.loads(s)
    except Exception:
        return None

# -----------------------------
# Stage 1: Build judge-ready samples (full + partial) for EVERY model
# -----------------------------
if not PHASE4_DIR.exists():
    raise FileNotFoundError(f"Phase 4 task directory not found: {PHASE4_DIR}")

model_dirs = [d for d in PHASE4_DIR.iterdir() if d.is_dir()]
if not model_dirs:
    raise FileNotFoundError(f"No model subdirectories found under {PHASE4_DIR}")

# discovery table
table = Table(title=f"Phase 5 — Discovery for Task: {TASK_NAME}")
table.add_column("Model Dir")
table.add_column("Latest samples_*.jsonl")
table.add_column("Latest results_*.json")
table.add_column("Judge samples exist?")
table.add_column("Judge partial exist?")
stage1_info = []

for mdir in model_dirs:
    spath = newest(mdir.glob("samples_*.jsonl"))
    rpath = newest(mdir.glob("results_*.json"))
    if not spath or not rpath:
        stage1_info.append((mdir, None, None, None, None))
        continue

    # derive deterministic names from the original samples basename – enables resume
    base = spath.stem  # e.g., "samples_sysengbench-osq_2025-10-01T03-58-29.887651"
    judge_stub = f"{base}_{mdir.name}_{JUDGE_MODEL.replace('/', '_')}-judge"
    judge_full   = SAMPLES_OUT / f"{judge_stub}.jsonl"
    judge_partial= SAMPLES_OUT / f"{judge_stub}-partial.jsonl"

    stage1_info.append((mdir, spath, rpath, judge_full, judge_partial))
    table.add_row(
        mdir.name,
        spath.name,
        rpath.name,
        "✅" if judge_full.exists() else "❌",
        "✅" if judge_partial.exists() else "❌",
    )

console.print(table)

# Create judge-ready files where missing; DO include all samples (or first N)
TEMPLATE = {
    "technical_accuracy": {"score": None, "justification": None},
    "conceptual_understanding": {"score": None, "justification": None},
    "completeness": {"score": None, "justification": None},
    "clarity_organization": {"score": None, "justification": None},
    "professional_relevance": {"score": None, "justification": None},
    "overall_score": None,
    "overall_assessment": None,
    "key_strengths": None,
    "improvement_areas": None,
}

for (mdir, spath, rpath, judge_full, judge_partial) in stage1_info:
    if not spath or not rpath:
        console.print(f"[yellow]Skipping Stage 1 for {mdir.name} — missing samples/results[/yellow]")
        continue

    if judge_full and judge_full.exists() and judge_partial and judge_partial.exists():
        console.print(f"[cyan]Stage 1 already prepared for {mdir.name}[/cyan]")
        continue

    raw_samples = load_jsonl(spath)
    if SAMPLE_N > 0:
        raw_samples = raw_samples[:SAMPLE_N]

    rows = []
    for i, s in enumerate(tqdm(raw_samples, desc=f"Stage 1: Build judge samples for {mdir.name}", unit="sample")):
        entry = {
            "sample_id": i,
            "osq_question":      s.get("osq_question", ""),
            "expected_answer":   s.get("expected_answer", ""),
            "student_response":  s.get("student_response", ""),
            "blooms_level":      s.get("blooms_level", "N/A"),
            "se_domain":         s.get("se_domain", "General Systems Engineering"),
            "judge_fields":      copy.deepcopy(TEMPLATE),
            # placeholders for transparency; will fill during Stage 2
            "judge_prompt": None,
            "judge_raw_output": None,
            "timestamp_created": datetime.now().isoformat(),
        }
        rows.append(entry)

    # Always generate/refresh full + partial with the same base contents
    write_jsonl(judge_full, rows)
    write_jsonl(judge_partial, rows)
    console.print(f"[green]Stage 1 ready for {mdir.name}[/green] -> {judge_full.name} & {judge_partial.name}")

# -----------------------------
# Stage 2: Incremental judging (resume from partial), with tqdm
# -----------------------------
def run_incremental_judging_for_model(mdir, spath, rpath, judge_full, judge_partial):
    # Load current partial (may include previous judgments)
    partial_rows = load_jsonl(judge_partial)
    # Build "latest by sample_id" view (so newest lines for a given id win if duplicates exist)
    latest = OrderedDict()
    for row in partial_rows:
        latest[row["sample_id"]] = row

    # make a set of already-judged IDs
    done_ids = set()
    for sid, row in latest.items():
        jf = row.get("judge_fields", {})
        if isinstance(jf.get("overall_score"), (int, float)):
            done_ids.add(sid)

    total = len(latest)
    to_do = [sid for sid in range(total) if sid not in done_ids]
    if not to_do:
        console.print(f"[cyan]{mdir.name}: nothing to judge (all {total} already done)[/cyan]")
    else:
        console.print(f"[green]{mdir.name}: {len(to_do)}/{total} to judge[/green]")

    # Open partial in append mode; we will append a new record for each newly-judged sample
    with open(judge_partial, "a", encoding="utf-8") as fout:
        for sid in tqdm(to_do, desc=f"Judging {mdir.name}", unit="sample"):
            row = latest[sid]
            prompt = JUDGE_PROMPT.format(
                osq_question=row["osq_question"],
                expected_answer=row["expected_answer"],
                student_response=row["student_response"],
                blooms_level=row.get("blooms_level", "N/A"),
                se_domain=row.get("se_domain", "General Systems Engineering"),
            )

            # Call judge model
            try:
                completion = client.chat.completions.create(
                    model=JUDGE_MODEL,
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                    messages=[
                        {"role": "system", "content": "You are an expert evaluator in systems engineering education."},
                        {"role": "user", "content": prompt}
                    ]
                )
                raw = completion.choices[0].message.content.strip()
                parsed = safe_parse_json(raw)
                if parsed is None:
                    # record the error but keep raw for debugging/cost transparency
                    row["judge_fields"] = copy.deepcopy(TEMPLATE)
                    row["judge_fields"]["overall_assessment"] = "ERROR: Invalid JSON"
                else:
                    # fill judge fields from parsed output
                    jf = copy.deepcopy(TEMPLATE)
                    # Move parsed fields into jf with type checks
                    def extract_score(d, key):
                        v = d.get(key)
                        if isinstance(v, dict):
                            return v.get("score")
                        return v
                    jf["technical_accuracy"]      = {"score": extract_score(parsed, "technical_accuracy"),      "justification": (parsed.get("technical_accuracy") or {}).get("justification")}
                    jf["conceptual_understanding"]= {"score": extract_score(parsed, "conceptual_understanding"),"justification": (parsed.get("conceptual_understanding") or {}).get("justification")}
                    jf["completeness"]            = {"score": extract_score(parsed, "completeness"),            "justification": (parsed.get("completeness") or {}).get("justification")}
                    jf["clarity_organization"]    = {"score": extract_score(parsed, "clarity_organization"),    "justification": (parsed.get("clarity_organization") or {}).get("justification")}
                    jf["professional_relevance"]  = {"score": extract_score(parsed, "professional_relevance"),  "justification": (parsed.get("professional_relevance") or {}).get("justification")}
                    jf["overall_score"]           = parsed.get("overall_score")
                    jf["overall_assessment"]      = parsed.get("overall_assessment")
                    jf["key_strengths"]           = parsed.get("key_strengths")
                    jf["improvement_areas"]       = parsed.get("improvement_areas")
                    row["judge_fields"] = jf

                row["judge_prompt"]    = prompt
                row["judge_raw_output"]= raw if parsed is None else raw
                row["timestamp_judged"]= datetime.now().isoformat()

            except Exception as e:
                row["judge_fields"] = copy.deepcopy(TEMPLATE)
                row["judge_fields"]["overall_assessment"] = f"ERROR: {str(e)}"
                row["judge_prompt"]    = prompt
                row["judge_raw_output"]= None
                row["timestamp_judged"]= datetime.now().isoformat()

            # append the *updated* row to the partial log
            fout.write(json.dumps(row) + "\n")
            fout.flush()

    # Recompute latest view after appends (so last write per sample_id wins)
    refreshed = OrderedDict()
    for r in load_jsonl(judge_partial):
        refreshed[r["sample_id"]] = r

    # Write a clean “final judged samples” file (one line per sample, fully populated where judged)
    base = judge_full.stem  # keep the same stem but mark as finalized
    final_samples_path = SAMPLES_OUT / f"{base}-FINAL.jsonl"
    write_jsonl(final_samples_path, list(refreshed.values()))
    console.print(f"[bold green]✓ Final judged samples written:[/bold green] {final_samples_path.name}")

    # Aggregate into results_*_...-judge.json (LM-Eval parity)
    results_meta = json.load(open(rpath, "r", encoding="utf-8"))
    aggregate = aggregate_scores(refreshed)

    judged_results = results_meta.copy()
    judged_results["judge_metadata"] = {
        "judge_model": JUDGE_MODEL,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "scale": "0–100 (five 20-point dimensions)",
        "timestamp": datetime.now().isoformat(),
        "samples_judged": sum(1 for r in refreshed.values() if isinstance(r.get("judge_fields", {}).get("overall_score"), (int, float))),
        "source_model": mdir.name,
        "partial_file": str(judge_partial),
        "final_samples_file": str(final_samples_path),
    }
    judged_results["results"] = {
        TASK_NAME: {
            "mean_scores": aggregate,
            "n_evaluated": judged_results["judge_metadata"]["samples_judged"],
            "evaluation_mode": "LLM-as-a-Judge"
        }
    }

    ts = datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
    final_results_path = RESULTS_OUT / f"results_{ts}_{mdir.name}_{JUDGE_MODEL.replace('/', '_')}-judge.json"
    with open(final_results_path, "w", encoding="utf-8") as f:
        json.dump(judged_results, f, indent=2)

    console.print(f"[bold green]✓ Final judge results written:[/bold green] {final_results_path.name}")

# -----------------------------
# Execute Stage 2 across all models discovered in Stage 1
# -----------------------------
for (mdir, spath, rpath, judge_full, judge_partial) in stage1_info:
    if not spath or not rpath:
        continue
    # Both files must exist from Stage 1
    if not judge_full.exists() or not judge_partial.exists():
        console.print(f"[yellow]Skipping Stage 2 for {mdir.name} — no judge files[/yellow]")
        continue
    run_incremental_judging_for_model(mdir, spath, rpath, judge_full, judge_partial)

console.print("[bold cyan]All done.[/bold cyan]")


OPENROUTER_API_KEY present: True

                                   Phase 5 — Discovery for Task: sysengbench-osq                                   
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Model Dir   ┃ Latest samples_*.jsonl   ┃ Latest results_*.json    ┃ Judge samples exist? ┃ Judge partial exist? ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3__27b │ samples_sysengbench-osq… │ results_2025-10-01T04-2… │ ❌                   │ ❌                   │
│ gemma3__4b  │ samples_sysengbench-osq… │ results_2025-10-01T03-5… │ ❌                   │ ❌                   │
└─────────────┴──────────────────────────┴──────────────────────────┴──────────────────────┴──────────────────────┘

Stage 1: Build judge samples for gemma3__27b:   0%|          | 0/2 [00:00<?, ?sample/s]

Stage 1 ready for gemma3__27b -> 
samples_sysengbench-osq_2025-10-01T04-26-37.597737_gemma3__27b_openai_gpt-5-judge.jsonl & 
samples_sysengbench-osq_2025-10-01T04-26-37.597737_gemma3__27b_openai_gpt-5-judge-partial.jsonl

Stage 1: Build judge samples for gemma3__4b:   0%|          | 0/2 [00:00<?, ?sample/s]

...

gemma3__4b: 2/2 to judge

Judging gemma3__4b:   0%|          | 0/2 [00:00<?, ?sample/s]

✓ Final judged samples written: 
samples_sysengbench-osq_2025-10-01T03-58-29.887651_gemma3__4b_openai_gpt-5-judge-FINAL.jsonl

✓ Final judge results written: results_2025-10-16T23-36-15_gemma3__4b_openai_gpt-5-judge.json

All done.

# NEED TO:
- Remove the "judge samples" and just have straight up judging process. too many "progress bars"
- need the new samples and results file to be in a directory that matches the task > model directory structure it was pulled from.
- Need the samples jsonl to ACTUALLY INCLUDE THE QUESTION AND ANSWER!